# TraceWhisperer 입문 종합 튜토리얼
## ChipWhisperer Husky + STM32F3 기반 실습

---

이 노트북은 **ChipWhisperer Husky**와 **CW308 + STM32F3** 타깃을 사용해
**TraceWhisperer를 처음 다루는 사용자**가 처음부터 끝까지 막힘 없이
따라갈 수 있도록 정리된 학습 자료입니다.

### 이 튜토리얼의 학습 목표
1. TraceWhisperer가 **무엇인지**, 왜 일반 ChipWhisperer 캡처에 더해서 사용하는지를 이해한다.
2. **SWO 모드**로 TraceWhisperer를 최소 설정으로 동작시킨다.
3. **펌웨어 트리거 + 패턴 매칭** 방식으로 trace event timestamp를 얻는다.
4. 얻은 timestamp를 **전력 파형 위에 오버레이**하여 시각화한다.
5. 펌웨어가 바뀌어 함수 주소가 달라졌을 때, **ELF에서 주소를 직접 찾아** 다시 매칭하는 방법을 익힌다.

### 이 튜토리얼의 대상
- ChipWhisperer 기본 캡처(`cw.capture_trace`)는 해 본 적이 있다.
- Husky와 STM32F3 타깃 연결은 할 수 있다.
- TraceWhisperer는 처음이거나, 공식 예제를 그대로 따라만 본 적이 있다.

### 요구 하드웨어 / 환경

| 항목 | 사양 |
|------|------|
| 캡처 보드 | ChipWhisperer Husky (또는 Husky Plus) |
| 타깃 보드 | CW308 + STM32F3 |
| 점퍼 케이블 | 최소 3개 (TMS / TCK / TDO 연결용) |
| 펌웨어 | `simpleserial-trace` (STM32F3 빌드) |
| 호스트 환경 | ChipWhisperer Jupyter 예제가 실행 가능한 상태 |

> ⚠️ 이 튜토리얼은 `simpleserial-trace` **전용** 펌웨어가 필요합니다.
> 일반 `simpleserial-aes` 펌웨어와는 다릅니다.

---

### 학습 흐름 (한눈에 보기)

```
[배경 지식] → [하드웨어 연결] → [설정값 모으기] → [Husky 연결]
   → [펌웨어 프로그래밍] → [SWO 인터페이스 활성화]
   → [캡처/매칭 조건 설정] → [PC 주소 등록]
   → [1회 캡처 + 시각화] → [다중 캡처 + 통계]
   → [(선택) ELF에서 주소 찾기] → [정리]
   → [문제 해결 체크리스트]
```


---
## 1. 사전 지식: TraceWhisperer는 왜 필요한가

### 전력 파형만 볼 때의 한계
일반 ChipWhisperer 캡처는 타깃의 **전력 파형(Power Trace)** 만 잡습니다.

```
기존 ChipWhisperer:    전력 파형만 캡처
                       → "지금 뭔가 하고 있는데... 어디가 SubBytes지?"

TraceWhisperer 추가:   전력 파형 + Arm 디버그 트레이스 이벤트 동시 캡처
                       → "SubBytes() 가 정확히 이 시점에 실행됐다!"
```

전력 파형만 보면 *어느 코드가 실행 중인지*를 추측에 의존해야 하지만,
TraceWhisperer를 더하면 **함수 단위로 시간 기준점**이 생깁니다.
이는 부채널 분석(SCA)에서 트레이스 정렬, 관심 구간 추출, 트리거 정밀화에 유용합니다.

### Arm CoreSight 디버그 트레이스 구성 요소

이 튜토리얼에서 알아야 할 핵심 블록은 다음과 같습니다.

| 구성요소 | 역할 | 이 튜토리얼에서의 사용 |
|----------|------|------------------------|
| **DWT** (Data Watchpoint and Trace) | PC 비교 / 이벤트 생성 | 함수 시작 주소와 PC가 일치하면 이벤트 발생 |
| **ETM / ITM** | 명령어/데이터 trace 패킷 생성 | DWT 이벤트를 trace 패킷으로 변환 |
| **TPI / SWO** (Trace Port Interface / Serial Wire Output) | trace 데이터 직렬 출력 | SWO 핀 1개로 trace 데이터 전송 |

### 트레이스 인터페이스: SWO vs 병렬

| 모드 | 핀 수 | 대역폭 | 이 튜토리얼 |
|------|-------|--------|--------------|
| **SWO** (Serial Wire Output) | 1개 | 낮음 | ✅ 사용 (STM32F3 지원) |
| **병렬 트레이스** | 4~5개 | 높음 | ✗ 별도 타깃 필요 |

### TraceWhisperer 핵심 트릭: FPGA 패턴 매칭

SWO 스트림은 TPIU 포맷이라는 복잡한 바이너리입니다.
이를 호스트 PC에서 소프트웨어 파싱하면 지연이 커서 실시간 트리거로 쓰기 어렵습니다.

TraceWhisperer는 **Husky 내부 FPGA**에서 패턴 매칭을 수행합니다.

- 미리 등록한 바이트 패턴이 SWO 스트림에 나타나는 순간을 감지
- 그 순간의 **타임스탬프**만 기록 → 매우 단순하고 정확함
- 소프트웨어 파싱 없이 하드웨어 레벨 처리

이 튜토리얼에서는 이 패턴 매칭 모드를 사용합니다.


---
## 2. 하드웨어 연결 확인 (가장 중요!)

> 💡 실습에서 가장 자주 발생하는 실수가 **점퍼선 미연결**입니다.
> 코드를 실행하기 전에 반드시 아래 연결 상태를 확인하세요.

### SWO 트레이스를 위한 점퍼 케이블 연결

CW308 보드의 JTAG/SWD 신호를 Husky의 USERIO 포트에 연결합니다.

```
STM32F3 타깃 (CW308)         Husky USERIO 포트
─────────────────────         ─────────────────
TMS  (JTAG 모드 선택)   ────►  D0
TCK  (JTAG 클럭)        ────►  D1
TDO  (트레이스 출력)    ────►  D2
```

CW308 보드에서 이 핀들의 위치:
- **TMS**: 20핀 커넥터의 핀 1 (또는 보드의 JTAG 헤더)
- **TCK**: 20핀 커넥터의 핀 9
- **TDO (SWO)**: 20핀 커넥터의 핀 3

> 💡 Husky와 CW308은 20핀 UFO 커넥터로 이미 연결되어 있습니다.
> USERIO 포트(D0~D2)만 별도 점퍼 케이블로 연결해 주면 됩니다.

### 연결 확인 체크리스트
- [ ] CW308과 Husky가 20핀 커넥터로 연결됨
- [ ] TMS → D0 점퍼 케이블 연결됨
- [ ] TCK → D1 점퍼 케이블 연결됨
- [ ] TDO → D2 점퍼 케이블 연결됨
- [ ] STM32F3에 `simpleserial-trace` 펌웨어가 준비되어 있음 (또는 본 노트북에서 프로그래밍할 예정)


---
## 3. 실습 설정값 한눈에 보기

이 튜토리얼에서 자주 바꾸게 될 값들을 **한 셀에 모아 둡니다.**
경로 / 주소 / 클럭 등은 환경에 따라 달라질 수 있으니, 본인 환경에 맞게 수정한 뒤
이후 셀들은 그대로 실행하면 됩니다.

> 📌 함수 시작 주소(`TRACE_MATCH_ADDR0`, `TRACE_MATCH_ADDR1`)는
> **펌웨어 빌드에 따라 달라집니다.** 처음 한 번은 아래 기본값으로 시도해 보고,
> 만약 trace event가 잡히지 않으면 **§13. ELF에서 주소 찾기** 절을 이용해
> 다시 입력하세요.


In [1]:
# ─────────────────────────────────────────────────────────────
# 플랫폼 / 경로 설정
# ─────────────────────────────────────────────────────────────
PLATFORM        = "CW308_STM32F3"   # 타깃 MCU
SCOPETYPE       = "OPENADC"         # Husky 사용 시 OPENADC
TRACE_PLATFORM  = "Husky"           # TraceWhisperer 플랫폼
TRACE_INTERFACE = "SWO"             # SWO 또는 parallel
HUSKY_SERIAL_NUMBER = "502032204c5846303030382032323037"

# simpleserial-trace 펌웨어 경로 (실제 디렉터리 구조에 맞게 수정)
FW_HEX_PATH = "simpleserial-trace/simpleserial-trace-CW308_STM32F3.hex"
FW_ELF_PATH = "simpleserial-trace/simpleserial-trace-CW308_STM32F3.elf"

# 이미 simpleserial-trace 가 올라가 있다면 False 로 두어도 됩니다.
DO_PROGRAM_TARGET = True

# ─────────────────────────────────────────────────────────────
# 캡처 / 트레이스 설정 (Husky 권장 시작값)
# ─────────────────────────────────────────────────────────────
ADC_SAMPLES = 31000   # AES 한 라운드 전체를 충분히 담을 길이
GAIN_DB     = 12      # STM32F3 타깃에 맞는 일반적인 게인값

TRACE_CLOCK_SOURCE   = "target_clock"   # 타깃 클럭을 trace 기준 클럭으로 사용
SWO_TRIGGER_FREQ_MUL = 8                # TraceWhisperer SWO 오버샘플링 배율
SWO_ACPR             = 0                # 0 = 가장 빠른 SWO 속도

# ─────────────────────────────────────────────────────────────
# 매칭할 함수 시작 주소
#
# 아래 값은 ChipWhisperer 공식 깃허브의 simpleserial-trace 펌웨어를
# CW308_STM32F3 타깃으로 빌드했을 때 기준입니다.
#
# 새로 빌드한 경우 simpleserial-trace-CW308_STM32F3.txt(.elf 디스어셈블 결과) 에서
# 다음 함수의 시작 주소를 확인해 아래 값을 갱신하세요.
#   - SubBytes()
#   - AddRoundKey()
#
# 본 튜토리얼에서 사용한 빌드 결과 예시:
#   SubBytes()    : 0x080017d8
#   AddRoundKey() : 0x080017a4
# ─────────────────────────────────────────────────────────────
TRACE_MATCH_ADDR0 = 0x080017d8   # SubBytes
TRACE_MATCH_ADDR1 = 0x080017a4   # AddRoundKey

# ─────────────────────────────────────────────────────────────
# 펌웨어 식별 힌트 (프로그래밍 후 검증용 문자열)
# ─────────────────────────────────────────────────────────────
EXPECTED_FW_HINT = "ChipWhisperer simpleserial-trace"

print("PLATFORM           =", PLATFORM)
print("FW_HEX_PATH        =", FW_HEX_PATH)
print("DO_PROGRAM_TARGET  =", DO_PROGRAM_TARGET)
print("TRACE_MATCH_ADDR0  = 0x{:08x}  (SubBytes 예상)".format(TRACE_MATCH_ADDR0))
print("TRACE_MATCH_ADDR1  = 0x{:08x}  (AddRoundKey 예상)".format(TRACE_MATCH_ADDR1))


PLATFORM           = CW308_STM32F3
FW_HEX_PATH        = simpleserial-trace/simpleserial-trace-CW308_STM32F3.hex
DO_PROGRAM_TARGET  = True
TRACE_MATCH_ADDR0  = 0x080017d8  (SubBytes 예상)
TRACE_MATCH_ADDR1  = 0x080017a4  (AddRoundKey 예상)


---
## 4. 라이브러리 임포트 및 지정 Husky Plus 연결

이 단계에서는 필요한 라이브러리를 불러오고 `HUSKY_SERIAL_NUMBER`에 지정된 장비만 직접 엽니다. TraceWhisperer 실습 장비가 SCA·FA 장비와 바뀌는 것을 막기 위해 연결 실패 시 다른 장비로 fallback하지 않습니다. 펌웨어가 사용하는 SimpleSerial 1.1에 맞춰 `SimpleSerial` 타겟을 생성한 뒤 Husky의 trace 블록을 활성화합니다.

### 실행 후 확인할 것
- `scope.trace.enabled = True`
- `scope.trace.target = target`
- `scope.gain.db`, `scope.adc.samples` 값이 정상 반영되었는지

In [2]:
import time
import re
import subprocess
import numpy as np
import chipwhisperer as cw

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span
from bokeh.resources import INLINE

output_notebook(INLINE)
print(f"ChipWhisperer 버전: {cw.__version__}")


/usr/local/lib/python3.12/site-packages/chipwhisperer/capture/trace/TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


Loading BokehJS ...

ChipWhisperer 버전: 6.0.0


In [3]:
# 지정된 TraceWhisperer 강의용 Husky Plus만 연다. 시리얼 없는 연결이나
# 다른 장비로의 fallback은 장비 역할을 뒤바꿀 수 있으므로 사용하지 않는다.
scope = cw.scope(sn=HUSKY_SERIAL_NUMBER)
target = cw.target(scope, cw.targets.SimpleSerial)
scope.default_setup()

# Husky의 trace 블록과 타겟 연결
scope.trace.target = target
scope.trace.enabled = True

# 전력 파형 캡처 파라미터
scope.adc.samples = ADC_SAMPLES
scope.gain.db = GAIN_DB

print(f'husky serial     = {HUSKY_SERIAL_NUMBER}')
print(f'adc.samples      = {scope.adc.samples}')
print(f'gain.db          = {scope.gain.db}')
print(f'trace.enabled    = {scope.trace.enabled}')
print(f'clkgen_freq      = {scope.clock.clkgen_freq/1e6:.2f} MHz')

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 327828                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

---
## 5. 리셋 및 상태 확인 헬퍼

TraceWhisperer 실습에서는 **리셋 후 다시 시도**가 매우 자주 필요합니다.
특히 다음 상황에서 리셋이 거의 필수입니다.

- 펌웨어 프로그래밍 직후
- SWO/JTAG 모드 전환 직후
- `capture_trace()` 가 `None` 을 반환할 때 (capture failed)
- `scope.trace.fifo_empty()` 가 `True` 일 때
- `scope.trace.get_fw_buildtime()` 결과가 비어 있거나 이상할 때

먼저 헬퍼 함수들을 정의해 둡니다. 이후 모든 단계에서 재사용합니다.


In [4]:
def reset_target(scope, delay=0.05):
    """스코프의 nRST 핀으로 타겟 MCU를 리셋한다.

    ``scope``는 쓰기 가능한 ``io.nrst``를 제공해야 하고, ``delay``는 low와
    high 상태를 각각 유지할 초 단위 시간이다. 반환값은 없다. 핀을 low에서
    high로 바꾸고 두 번 대기하므로 타겟 상태와 실행 시간이 달라진다. 장치 연결
    또는 속성 쓰기에 실패하면 해당 하드웨어 예외가 호출자에게 전달된다.
    """
    scope.io.nrst = "low"
    time.sleep(delay)
    scope.io.nrst = "high"
    time.sleep(delay)


def get_fw_buildtime_safe(scope):
    """TraceWhisperer 펌웨어 빌드 시각을 읽어 문자열로 반환한다.

    ``scope.trace.get_fw_buildtime()``의 반환값을 그대로 돌려준다. 읽기 중 발생한
    모든 예외는 전파하지 않고 원인을 포함한 ``<fw_buildtime read failed: ...>``
    문자열로 바꾼다. 하드웨어 설정은 변경하지 않는다.
    """
    try:
        return scope.trace.get_fw_buildtime()
    except Exception as e:
        return f"<fw_buildtime read failed: {e}>"


def print_basic_status(scope):
    """펌웨어와 FPGA 빌드 정보를 표준 출력에 기록한다.

    ``scope``에서 펌웨어와 FPGA 정보를 읽으며 반환값은 없다. 펌웨어 읽기 실패는
    오류 문자열로, FPGA 속성 읽기 실패는 ``<unavailable>``과 예외 내용으로
    출력한다. 장치 상태를 바꾸지 않지만 표준 출력이라는 부작용이 있다.
    """
    print("FW buildtime :", get_fw_buildtime_safe(scope))
    try:
        print("Husky FPGA   :", scope.fpga_buildtime)
    except Exception as e:
        print("Husky FPGA   : <unavailable>", e)


reset_target(scope)
print_basic_status(scope)


FW buildtime : ChipWhisperer simpleserial-trace, compiled Aug 25 2026, 22:01:00
Husky FPGA   : 12/16/2024, 13:17


---
## 6. (선택) 펌웨어 빌드

이미 `simpleserial-trace-CW308_STM32F3.hex` 파일이 준비되어 있다면 이 셀은 **건너뛰어도 됩니다.**

직접 빌드를 원하거나, 함수 주소를 ELF에서 찾아 다시 매칭할 계획이라면 이 셀을 실행하세요.
빌드 결과로 `.elf`, `.hex`, 그리고 디스어셈블 결과가 담긴 `.txt` 파일이 만들어집니다.

> ⚠️ 빌드 환경이 갖춰져 있어야 합니다 (arm-none-eabi 툴체인 + make).


In [5]:
%%bash -s "$PLATFORM"

cd simpleserial-trace/
make PLATFORM=$1 clean
make PLATFORM=$1

f=simpleserial-trace-CW308_STM32F3.elf

# 디스어셈블 결과 저장 (함수 시작 주소 확인용)
arm-none-eabi-objdump -D -x -s -S -l "$f" > "${f%.elf}.txt"


No CRYPTO_TARGET passed - defaulting to TINYAES128C
Building for platform CW308_STM32F3 with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
rm -f -- simpleserial-trace-CW308_CC2538.hex simpleserial-trace-CW301_AVR.hex simpleserial-trace-CW303.hex simpleserial-trace-CW304.hex simpleserial-trace-CW308_MEGARF.hex simpleserial-trace-CW308_SAM4L.hex simpleserial-trace-CW308_STM32F0.hex simpleserial-trace-CW308_STM32F1.hex simpleserial-trace-CW308_STM32F2.hex simpleserial-trace-CW308_STM32F3.hex simpleserial-trace-CW308_STM32F4.hex simpleserial-trace-CW308_K24F.hex simpleserial-trace-CW308_NRF52.hex simpleserial-trace-CW308_AURIX.hex simpleserial-trace-CW308_SAML11.hex simpleserial-trace-CW308_EFM32TG11B.hex simpleserial-trace-CWLITEARM.hex simpleserial-trace-CWLITEXMEGA.hex simpleserial-trace-CWNANO.hex simpleserial-trace-CWHUSKY.hex simpleser

---
## 7. 타깃 펌웨어 프로그래밍

`simpleserial-trace` 펌웨어를 STM32F3 타깃에 올립니다.
이 펌웨어는 일반 `simpleserial-aes` 와 기능은 같지만, **SWO 트레이스 핀을 초기화하는 코드가 추가**되어 있습니다.

### 이 셀을 실행해야 하는 경우
- `get_fw_buildtime()` 결과에 `ChipWhisperer simpleserial-trace` 가 보이지 않을 때
- 캡처가 반복적으로 실패할 때
- 실습 환경을 처음 세팅할 때

이미 올바른 펌웨어가 올라가 있다면 §3 의 `DO_PROGRAM_TARGET = False` 로 두면 됩니다.


In [6]:
if DO_PROGRAM_TARGET:
    print("Programming target firmware...")
    prog = cw.programmers.STM32FProgrammer
    cw.program_target(scope, prog, FW_HEX_PATH)
    reset_target(scope)
    time.sleep(0.1)
    print("✅ 펌웨어 프로그래밍 완료")
else:
    print("DO_PROGRAM_TARGET = False → 프로그래밍 건너뜀")

print_basic_status(scope)

fw_info = get_fw_buildtime_safe(scope)
assert EXPECTED_FW_HINT in str(fw_info), (
    "올바른 simpleserial-trace 펌웨어가 아닌 것 같습니다.\n"
    f"  현재 응답: {fw_info!r}\n"
    f"  FW_HEX_PATH 를 확인하거나 DO_PROGRAM_TARGET=True 로 다시 시도하세요."
)
print(f"✅ 펌웨어 식별 OK: {fw_info}")


Programming target firmware...
Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 7227 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 7227 bytes
✅ 펌웨어 프로그래밍 완료
FW buildtime : ChipWhisperer simpleserial-trace, compiled Aug 25 2026, 22:09:29
Husky FPGA   : 12/16/2024, 13:17
✅ 펌웨어 식별 OK: ChipWhisperer simpleserial-trace, compiled Aug 25 2026, 22:09:29


---
## 8. SWO 트레이스 인터페이스 설정

이 단계가 TraceWhisperer 설정의 핵심입니다.
Arm 프로세서는 리셋 시 **JTAG 모드**로 시작하는데, SWO 트레이스를 사용하려면 **SWD 모드**로 전환해야 합니다.

### 이 단계에서 하는 일 (4가지)
1. Front-end clock source 선택 (`target_clock`)
2. SWO 모드로 전환 (`trace_mode = "SWO"`)
3. JTAG → SWD 전환 (`jtag_to_swd()`)
4. SWO 클럭 파라미터 설정 + lock 확인

### SWO 클럭 파라미터의 의미

```
SWO 비트 주기      = (TPI.ACPR + 1) × 타깃 클럭 주기
TraceWhisperer SWO 복원 클럭 = 타깃 클럭 × trigger_freq_mul
swo_div            = trigger_freq_mul × (ACPR + 1)
```

- `ACPR = 0`: 가장 빠른 SWO 속도 (타깃 클럭과 동일)
- `trigger_freq_mul = 8`: TraceWhisperer 가 SWO 클럭의 8배로 오버샘플링 → 안정적 복원

### 실행 후 가장 중요하게 볼 값
- `scope.trace.clock.fe_clock_alive`  (True 여야 함)
- `scope.trace.clock.swo_clock_locked`  (True 여야 함)
- USERIO D2 핀이 HIGH 인지 (SWD 활성화 시 SWO 라인이 idle high 를 유지함)


In [7]:
assert TRACE_INTERFACE == "SWO", "이 입문 예제는 SWO 기준으로 작성되었습니다."

# ── 1) Front-end clock source 선택 ────────────────────────────
scope.trace.clock.fe_clock_src = TRACE_CLOCK_SOURCE
assert scope.trace.clock.fe_clock_alive, (
    "선택한 front-end clock 이 살아 있지 않습니다.\n"
    " → HS2 핀 / 타깃 클럭 출력 / 타깃 전원을 확인하세요."
)
print(f"✅ fe_clock_src = {scope.trace.clock.fe_clock_src} (alive)")

# ── 2) SWO 모드 전환 ─────────────────────────────────────────
scope.trace.trace_mode = "SWO"

# ── 3) JTAG → SWD 전환 ───────────────────────────────────────
# TMS/TCK 핀에 특수 시퀀스를 보내 프로세서를 SWD 모드로 바꾼다.
scope.trace.jtag_to_swd()
print("✅ JTAG → SWD 전환 완료")

# ── 4) SWO 클럭 파라미터 설정 ─────────────────────────────────
scope.trace.clock.swo_clock_freq = scope.clock.clkgen_freq * SWO_TRIGGER_FREQ_MUL
scope.trace.target_registers.TPI_ACPR = SWO_ACPR
scope.trace.swo_div = SWO_TRIGGER_FREQ_MUL * (SWO_ACPR + 1)

assert scope.trace.clock.swo_clock_locked, (
    "SWO clock lock 실패.\n"
    " → 점퍼 케이블 연결 / 클럭 설정 / 타깃 리셋을 다시 확인하세요."
)
print(f"✅ swo_clock_locked = {scope.trace.clock.swo_clock_locked}")
print(f"   ACPR             = {SWO_ACPR}")
print(f"   오버샘플링 배율  = {SWO_TRIGGER_FREQ_MUL}")
print(f"   SWO clock freq   = {scope.trace.clock.swo_clock_freq/1e6:.2f} MHz")
print(f"   swo_div          = {scope.trace.swo_div}")

# ── 5) SWO 라인 상태 확인 (USERIO D2) ─────────────────────────
# Husky 버전에 따라 userio 접근 방식이 조금 다를 수 있어 둘 다 시도한다.
swo_line_ok = None
try:
    swo_line_ok = bool(scope.userio.pins[2].status)
except Exception:
    try:
        swo_line_ok = bool(scope.userio.status & 0x4)
    except Exception:
        swo_line_ok = None

if swo_line_ok:
    print("✅ SWO 라인 정상 (HIGH)  — SWD 활성화됨")
else:
    print("⚠️  SWO 라인이 HIGH 가 아닙니다.")
    print("   TDO → D2 점퍼선 / 타깃 리셋을 확인 후, 필요하면 다음 셀을 실행하세요.")


✅ fe_clock_src = target_clock (alive)
✅ JTAG → SWD 전환 완료
✅ swo_clock_locked = True
   ACPR             = 0
   오버샘플링 배율  = 8
   SWO clock freq   = 58.91 MHz
   swo_div          = 8
✅ SWO 라인 정상 (HIGH)  — SWD 활성화됨


SWD 모드 전환 후 일부 타깃은 일시적으로 응답 불능 상태가 될 수 있습니다.
다음 셀로 응답을 한 번 더 확인합니다. 응답이 없으면 자동 리셋 후 재시도합니다.


In [8]:
fw_buildtime = get_fw_buildtime_safe(scope)
if not fw_buildtime or "failed" in str(fw_buildtime):
    print("타깃이 응답하지 않습니다. 리셋 후 재시도합니다...")
    reset_target(scope)
    time.sleep(0.1)
    fw_buildtime = get_fw_buildtime_safe(scope)

print(f"FW buildtime: {fw_buildtime}")


FW buildtime: ChipWhisperer simpleserial-trace, compiled Aug 25 2026, 22:09:29


---
## 9. 캡처 모드 / 패턴 매칭 / 트리거 설정

TraceWhisperer 의 캡처 모드는 두 가지가 있습니다.

| 모드 | 동작 | 장점 | 단점 |
|------|------|------|------|
| **Raw 캡처** | 모든 TraceWhisperer 패킷을 원시 바이트로 저장 | 상세 분석 가능 | 호스트 파싱 필요, 데이터량 큼 |
| **패턴 매칭** | 특정 바이트 패턴이 보일 때 **타임스탬프만** 저장 | 단순, 실시간 처리 | 사전 정의된 패턴만 감지 |

이 튜토리얼에서는 **패턴 매칭 모드**를 사용합니다.
DWT_COMP 이벤트 패킷의 헤더 바이트 `[3, 8, 32]` 를 패턴으로 등록하면
PC 일치 이벤트가 발생할 때마다 그 시점이 정확히 기록됩니다.

### 패턴 `[3, 8, 32]` 의 의미
- `3`  = SWIT 패킷 소스 ID
- `8`  = 비교 이벤트 타입
- `32` = PC 일치 이벤트 코드

### 트리거 소스 / 캡처 지속 조건
- `trigger_source = "firmware trigger"`: TIO4 핀의 펌웨어 트리거로 캡처 시작 (가장 안정적)
- `mode = "while_trig"`: 트리거가 HIGH 인 동안만 캡처 → 전력 파형과 시간축이 자연스럽게 정렬


In [9]:
# ── DWT_CTRL: 주기적 sync 프레임 비활성화 ─────────────────────
# 0x40000021 = CYCCNTENA(bit0) + EXCTRCENA(bit16) + NOPROFTRAP(bit30)
#  → 입문 단계에서는 sync 프레임이 timing 분석을 방해할 수 있어 끈다.
#
# 주의: ChipWhisperer 버전에 따라 register 읽기가
#       int 또는 hex 문자열('40000021') 을 돌려준다.
#       두 경우 모두에서 동작하도록 아래 helper 를 사용한다.
def reg_to_int(val):
    """register read 결과를 int 로 정규화."""
    if isinstance(val, int):
        return val
    if isinstance(val, str):
        s = val.strip().lower()
        if s.startswith("0x"):
            s = s[2:]
        return int(s, 16)
    raise TypeError(f"unexpected register type: {type(val)}")

before = reg_to_int(scope.trace.target_registers.DWT_CTRL)
print(f"DWT_CTRL (변경 전) = 0x{before:08x}")

scope.trace.target_registers.DWT_CTRL = 0x40000021

after = reg_to_int(scope.trace.target_registers.DWT_CTRL)
print(f"DWT_CTRL (변경 후) = 0x{after:08x}")


DWT_CTRL (변경 전) = 0x400007f1
DWT_CTRL (변경 후) = 0x40000021


In [10]:
# ── 캡처 모드 / 패턴 / 트리거 설정 ────────────────────────────
# 1) raw=False  → pattern match 모드
scope.trace.capture.raw = False

# 2) DWT_COMP PC 일치 이벤트의 헤더 패턴
scope.trace.set_pattern_match(0, [3, 8, 32])

# 3) rule 0 만 활성화
scope.trace.capture.rules_enabled = [0]

# 4) 트리거 소스: 펌웨어 TIO4 트리거
scope.trace.capture.trigger_source = "firmware trigger"

# 5) 캡처 지속 조건: 트리거 HIGH 동안만
scope.trace.capture.mode = "while_trig"

print("✅ 캡처/매칭/트리거 설정 완료")
print(f"   capture.raw         = {scope.trace.capture.raw}")
print(f"   rules_enabled       = {scope.trace.capture.rules_enabled}")
print(f"   trigger_source      = {scope.trace.capture.trigger_source}")
print(f"   capture.mode        = {scope.trace.capture.mode}")


✅ 캡처/매칭/트리거 설정 완료
   capture.raw         = False
   rules_enabled       = [0]
   trigger_source      = firmware trigger
   capture.mode        = while_trig


---
## 10. PC 주소 매칭 설정

DWT (Data Watchpoint and Trace) 의 비교기 (COMP0, COMP1) 에 **감시할 함수의 시작 주소**를 등록합니다.
PC(Program Counter)가 해당 주소를 지나갈 때마다 TraceWhisperer event(이벤트)가 발생합니다.

이 튜토리얼에서는 AES 의 두 핵심 함수를 대상으로 둡니다.
- `SubBytes`     — S-Box 비선형 치환 단계 (전력 분석 공격의 주요 타깃)
- `AddRoundKey`  — 라운드 키와의 XOR

### 주의: 함수 주소는 빌드에 따라 달라집니다
§3 의 `TRACE_MATCH_ADDR0`, `TRACE_MATCH_ADDR1` 은 **이 튜토리얼이 사용한 빌드 결과** 기준입니다.
- 새로 빌드했거나
- 다른 컴파일 옵션으로 만들었거나
- 펌웨어 버전이 다르면

**주소가 달라지므로** §13 의 ELF 주소 찾기 절차를 사용해 주세요.

### 주기적 PC 샘플링은 끈다
DWT 는 두 가지 방식으로 PC 정보를 출력할 수 있습니다.
1. 주기적 샘플링 (일정 간격마다 현재 PC 출력)
2. 비교 이벤트 (COMP 와 일치할 때만 출력)  ← **본 튜토리얼이 사용**

두 가지가 동시에 켜지면 SWO 대역폭을 낭비하고 이벤트가 섞이므로 주기적 샘플링은 끕니다.


In [11]:
# DWT.COMP0 / DWT.COMP1 에 주소 등록
scope.trace.set_isync_matches(
    addr0=TRACE_MATCH_ADDR0,
    addr1=TRACE_MATCH_ADDR1,
    match="both",   # 두 주소 모두에서 이벤트 발생
)

# 입문 예제에서는 periodic PC sampling 을 끈다.
scope.trace.set_periodic_pc_sampling(enable=0)

print("✅ PC 주소 매칭 설정 완료")
print(f"   COMP0 (SubBytes 예상)    = 0x{TRACE_MATCH_ADDR0:08x}")
print(f"   COMP1 (AddRoundKey 예상) = 0x{TRACE_MATCH_ADDR1:08x}")
print("   match='both' / periodic PC sampling = OFF")


✅ PC 주소 매칭 설정 완료
   COMP0 (SubBytes 예상)    = 0x080017d8
   COMP1 (AddRoundKey 예상) = 0x080017a4
   match='both' / periodic PC sampling = OFF


---
## 11. 재현 가능한 1회 캡처 함수 정의

여러 번 캡처할 일이 많기 때문에, **재현 가능한 1회 캡처 절차**를 함수로 정리합니다.

이 함수가 하는 일은 다음과 같습니다.
1. (선택) 타깃 리셋
2. TraceWhisperer arm
3. 전력 파형 + 트리거 + trace 동시 캡처 (`cw.capture_trace`)
4. trace FIFO 비어 있지 않은지 확인
5. raw 데이터 읽기 + 패턴 매칭 timestamp 추출


In [12]:
def capture_once(scope, target, text=None, key=None,
                 reset_before=False, verbose=True):
    """TraceWhisperer 1회 캡처를 수행한다.

    Returns
    -------
    powertrace : ChipWhisperer Trace 객체 (전력 파형, 평문/암호문 포함)
    raw        : TraceWhisperer 가 읽은 내부 raw 데이터
    times      : 패턴 매칭 timestamp 리스트 — [(cycle, ...), ...]
    """
    if reset_before:
        reset_target(scope)
        time.sleep(0.1)

    if text is None:
        text = bytearray(16)
    if key is None:
        key = bytearray(16)

    # 1) TraceWhisperer 준비
    scope.trace.arm_trace()

    # 2) 전력 + trace 동시 캡처
    powertrace = cw.capture_trace(scope, target, text, key)
    assert powertrace is not None, (
        "capture_trace() 실패.\n"
        " → 타깃 응답, 케이블 연결, 펌웨어, 리셋 상태를 확인하세요."
    )

    # 3) trace FIFO 검증
    assert not scope.trace.fifo_empty(), (
        "TraceWhisperer FIFO 가 비어 있습니다.\n"
        " → arm_trace() 타이밍 / SWO 설정 / 함수 주소 / trigger 구간을 확인하세요."
    )

    # 4) raw + pattern match timestamp 추출
    raw   = scope.trace.read_capture_data()
    times = scope.trace.get_rule_match_times(raw, rawtimes=False, verbose=verbose)

    return powertrace, raw, times


print("✅ capture_once() 정의 완료")


✅ capture_once() 정의 완료


---
## 12. 첫 번째 캡처 — 실행 + 시각화

이제 실제로 1회 캡처를 수행하고, 결과를 **전력 파형 위에 trace 이벤트를 오버레이**해서 봅니다.
입문자가 가장 좋아하는 구간입니다.

### 기대하는 결과
- `powertrace.wave` : 전력 파형 (NumPy array)
- `raw`             : TraceWhisperer 가 읽은 내부 raw 데이터
- `times`           : 패턴 매칭 timestamp 리스트 (정상이라면 1개 이상)

### AES-128 의 예상 이벤트 수
AES-128 은 10 라운드입니다. SubBytes / AddRoundKey 가 각 라운드에서 호출되지만
마지막 라운드는 코드 경로가 약간 달라, 일반적으로 **약 21개의 이벤트**가 감지됩니다
(10 라운드 × 2 함수 + 마지막 1번의 AddRoundKey = 21).


In [13]:
powertrace, raw, times = capture_once(
    scope=scope,
    target=target,
    text=bytearray(16),
    key=bytearray(16),
    reset_before=False,
    verbose=True,
)

print(f"\n✅ trace event 총 {len(times)} 개 감지")
print("times (앞 10개) =", times[:10])
print("matched_pattern_data   =", scope.trace.capture.matched_pattern_data)
print("matched_pattern_counts =", scope.trace.capture.matched_pattern_counts)
print("\n💡 AES-128 기준으로 약 21개 이벤트가 정상입니다.")


     158 rule # 0, delta = 158
     337 rule # 0, delta = 179
     831 rule # 0, delta = 494
    1011 rule # 0, delta = 180
    1505 rule # 0, delta = 494
    1685 rule # 0, delta = 180
    2179 rule # 0, delta = 494
    2359 rule # 0, delta = 180
    2853 rule # 0, delta = 494
    3033 rule # 0, delta = 180
    3527 rule # 0, delta = 494
    3707 rule # 0, delta = 180
    4201 rule # 0, delta = 494
    4381 rule # 0, delta = 180
    4875 rule # 0, delta = 494
    5055 rule # 0, delta = 180
    5549 rule # 0, delta = 494
    5729 rule # 0, delta = 180
    6223 rule # 0, delta = 494
    6401 rule # 0, delta = 178
    6638 rule # 0, delta = 237

✅ trace event 총 21 개 감지
times (앞 10개) = [[158, 0], [337, 0], [831, 0], [1011, 0], [1505, 0], [1685, 0], [2179, 0], [2359, 0], [2853, 0], [3033, 0]]
matched_pattern_data   = 0008840135030820
matched_pattern_counts = [252, 0, 0, 0, 0, 0, 0, 0]

💡 AES-128 기준으로 약 21개 이벤트가 정상입니다.


### 타임스탬프 → 전력 Trace x축 변환

전력 Trace의 x축 단위(ADC Sample)와 TraceWhisperer 타임스탬프 단위(타깃 클럭)는 다릅니다.

```
trace timestamp 단위 : 타깃 클럭
ADC 샘플 단위        : 타깃 클럭 × adc_mul (Husky 의 오버샘플링 배율)

→ 전력 Trace x좌표 = TraceWhisperer 타임스탬프 × adc_mul
```

`get_adc_multiplier(scope)`는 Husky에서는 `scope.clock.adc_mul`을 반환하고, 다른 보드에서는 `scope.clock.adc_src`가 `clkgen_x4` 또는 `extclk_x4`이면 4, 그 밖에는 1을 반환합니다. 이 환산은 타깃 클럭과 ADC가 같은 기준 클럭에 동기화되었다는 전제에서 표시 좌표를 계산합니다.


In [14]:
def get_adc_multiplier(scope):
    """TraceWhisperer 타임스탬프를 ADC Sample 인덱스로 바꾸는 배율을 반환한다.

    Husky는 실제 ``adc_mul`` 값을 반환한다. 다른 보드는 ``adc_src``의
    x4 표기만 판별하며, 알 수 없는 소스는 1로 처리한다. 이 fallback은 예외를
    내지 않는 대신 비동기 클럭이나 다른 배율을 정확히 표현하지 못할 수 있다.
    스코프 상태를 읽기만 하며 하드웨어 설정은 바꾸지 않는다.
    """
    # Husky는 ADC 배율을 수치로 직접 노출한다.
    if getattr(scope, "_is_husky", False):
        return scope.clock.adc_mul
    # 다른 보드는 x4로 명시된 동기 클럭 소스만 판별할 수 있다.
    adc_src = getattr(scope.clock, "adc_src", None)
    if adc_src in ("clkgen_x4", "extclk_x4"):
        return 4
    return 1


multiplier = get_adc_multiplier(scope)
print(f"ADC 오버샘플링 배율 = {multiplier}×")
print(f"타깃 클럭          = {scope.clock.clkgen_freq/1e6:.2f} MHz")
print(f"ADC 샘플링 속도    = {scope.clock.clkgen_freq * multiplier / 1e6:.2f} MHz")

if times:
    print("\n앞쪽 5개 이벤트의 좌표 변환:")
    for i, t in enumerate(times[:5]):
        print(f"  event {i+1}: {t[0]:>8,} cycles → x = {t[0]*multiplier:>8,}")


ADC 오버샘플링 배율 = 4×
타깃 클럭          = 7.36 MHz
ADC 샘플링 속도    = 29.45 MHz

앞쪽 5개 이벤트의 좌표 변환:
  event 1:      158 cycles → x =      632
  event 2:      337 cycles → x =    1,348
  event 3:      831 cycles → x =    3,324
  event 4:    1,011 cycles → x =    4,044
  event 5:    1,505 cycles → x =    6,020


In [15]:
# ─────────────────────────────────────────────────────────────
# Bokeh 시각화: 전력 파형 + trace event 오버레이
#  - 빨간 선 : 전력 파형
#  - 검은 수직선 : SubBytes / AddRoundKey 호출 시점
# ─────────────────────────────────────────────────────────────
p = figure(
    width=1200,
    height=400,
    title="Power trace + TraceWhisperer event timestamps",
    x_axis_label="ADC sample index",
    y_axis_label="Power (normalized)",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

xrange = list(range(len(powertrace.wave)))
p.line(xrange, powertrace.wave,
       line_color="red", line_width=1, alpha=0.85,
       legend_label="Power trace")

vlines = []
for t in times:
    vlines.append(Span(
        location=t[0] * multiplier,
        dimension="height",
        line_color="black",
        line_width=1.5,
        line_alpha=0.7,
    ))
p.renderers.extend(vlines)

p.legend.location = "top_right"
show(p)

print(f"총 {len(times)} 개의 trace event 가 파형에 표시되었습니다.")
print("💡 마우스 휠로 확대하면 함수 호출 시점이 더 잘 보입니다.")


총 21 개의 trace event 가 파형에 표시되었습니다.
💡 마우스 휠로 확대하면 함수 호출 시점이 더 잘 보입니다.


---
## 13. 다중 캡처 + 통계 분석

실제 부채널 분석에서는 여러 Trace(트레이스)를 모읍니다. 입문 단계에서도
**여러 번 캡처해서 이벤트 위치가 얼마나 일관적인지** 보는 것이 매우 유용합니다.
타이밍이 흔들린다면 SWO 설정에 문제가 있을 가능성이 큽니다.


In [16]:
try:
    from tqdm.notebook import trange
except ImportError:
    def trange(n, desc=""):
        """tqdm이 없을 때 ``range(n)``을 반환하는 진행 표시 대체 함수다.

        ``n``은 반복 횟수이며 ``desc``는 API 호환용이라 사용하지 않는다.
        ``range``와 같은 오류 조건을 가지며 별도 출력이나 상태 변경은 없다.
        """
        return range(n)

NUM_TRACES = 10  # 실습용 소량. 실제 분석에는 수천 개 사용.

ktp = cw.ktp.Basic()

powertraces_list = []
trace_times_list = []
plaintexts = []
keys = []

print(f"{NUM_TRACES} 개 캡처 시작...\n")

for i in trange(NUM_TRACES, desc="capturing"):
    key, text = ktp.next()
    try:
        pt, raw_i, times_i = capture_once(
            scope, target,
            text=text, key=key,
            reset_before=False, verbose=False,
        )
    except AssertionError as e:
        print(f"  [{i}] 실패: {e}")
        continue

    powertraces_list.append(pt)
    trace_times_list.append(times_i)
    plaintexts.append(pt.textin)
    keys.append(pt.key)

print(f"\n✅ 캡처 완료: {len(powertraces_list)}/{NUM_TRACES} 성공")


10 개 캡처 시작...



capturing:   0%|          | 0/10 [00:00<?, ?it/s]


✅ 캡처 완료: 10/10 성공


In [17]:
# ── 이벤트 수 / 첫 이벤트 타이밍 통계 ─────────────────────────
event_counts = [len(t) for t in trace_times_list]

print("=" * 50)
print("이벤트 타이밍 통계")
print("=" * 50)
print(f"평균 이벤트 수: {np.mean(event_counts):.1f}")
print(f"최소 / 최대   : {min(event_counts)} / {max(event_counts)}")

EXPECTED_COUNT = 21  # AES-128 일반적 기대치
valid_indices = [i for i, c in enumerate(event_counts) if c == EXPECTED_COUNT]
print(f"\n유효 캡처 (이벤트 {EXPECTED_COUNT}개): {len(valid_indices)}/{len(powertraces_list)}")

if valid_indices:
    first_event_times = np.array(
        [trace_times_list[i][0][0] for i in valid_indices], dtype=float
    )
    print(f"\n첫 이벤트 timestamp 통계 (단위: 타깃 클럭 사이클)")
    print(f"  평균       : {first_event_times.mean():,.1f}")
    print(f"  표준편차   : {first_event_times.std():.2f}")
    print(f"  범위       : {first_event_times.min():,.0f} ~ {first_event_times.max():,.0f}")
    if first_event_times.std() < 2:
        print("\n✅ 첫 이벤트 timing 매우 안정적 (σ < 2 cycles)")
    else:
        print("\n⚠️ timing 변동이 있습니다. SWO 설정을 점검하세요.")


이벤트 타이밍 통계
평균 이벤트 수: 21.0
최소 / 최대   : 21 / 21

유효 캡처 (이벤트 21개): 10/10

첫 이벤트 timestamp 통계 (단위: 타깃 클럭 사이클)
  평균       : 158.0
  표준편차   : 0.00
  범위       : 158 ~ 158

✅ 첫 이벤트 timing 매우 안정적 (σ < 2 cycles)


In [18]:
# ── 여러 파형 오버레이 시각화 ─────────────────────────────────
from bokeh.palettes import Category10

p2 = figure(
    width=1200,
    height=450,
    title=f"다중 캡처 오버레이 ({min(5, len(valid_indices))}개)",
    x_axis_label="ADC sample index",
    y_axis_label="Power",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

if valid_indices:
    colors = Category10[10]
    display_count = min(5, len(valid_indices))
    for idx, vi in enumerate(valid_indices[:display_count]):
        pt = powertraces_list[vi]
        p2.line(
            list(range(len(pt.wave))), pt.wave,
            line_color=colors[idx % len(colors)],
            line_width=1, alpha=0.6,
            legend_label=f"trace {vi+1}",
        )

    # 첫 유효 캡처의 trace event 만 수직선으로 표시
    for t in trace_times_list[valid_indices[0]]:
        p2.renderers.append(Span(
            location=t[0] * multiplier, dimension="height",
            line_color="black", line_width=1, line_alpha=0.5,
        ))

    p2.legend.location = "top_right"
    p2.legend.click_policy = "hide"
    show(p2)
    print("💡 범례를 클릭하면 개별 파형을 숨기거나 표시할 수 있습니다.")
else:
    print("유효 캡처가 없습니다. 먼저 §13 위 셀의 결과를 확인하세요.")


💡 범례를 클릭하면 개별 파형을 숨기거나 표시할 수 있습니다.


In [19]:
# ── AES 라운드별 이벤트 간격 살펴보기 ─────────────────────────
if valid_indices:
    sample_times = trace_times_list[valid_indices[0]]
    print("=" * 60)
    print("AES 라운드별 이벤트 분석 (첫 유효 캡처 기준)")
    print("=" * 60)
    print(f"{'event':>6}  {'timestamp':>12}  {'간격(cyc)':>12}")
    print("-" * 40)
    prev = None
    intervals = []
    for i, t in enumerate(sample_times):
        ts = t[0]
        if prev is None:
            interval_str = "-"
        else:
            d = ts - prev
            intervals.append(d)
            interval_str = f"{d:>12,}"
        print(f"{i+1:>6}  {ts:>12,}  {interval_str}")
        prev = ts
    if intervals:
        avg = np.mean(intervals)
        print(f"\n평균 이벤트 간격: {avg:,.0f} cycles")
        print(f"  ≈ {avg / scope.clock.clkgen_freq * 1e6:.2f} μs @ {scope.clock.clkgen_freq/1e6:.2f} MHz")


AES 라운드별 이벤트 분석 (첫 유효 캡처 기준)
 event     timestamp       간격(cyc)
----------------------------------------
     1           158  -
     2           337           179
     3           831           494
     4         1,011           180
     5         1,505           494
     6         1,685           180
     7         2,179           494
     8         2,359           180
     9         2,853           494
    10         3,033           180
    11         3,527           494
    12         3,707           180
    13         4,201           494
    14         4,381           180
    15         4,875           494
    16         5,055           180
    17         5,549           494
    18         5,729           180
    19         6,223           494
    20         6,401           178
    21         6,638           237

평균 이벤트 간격: 324 cycles
  ≈ 44.00 μs @ 7.36 MHz


---
## 14. (선택·실용) 펌웨어를 새로 빌드했다면 — ELF 에서 함수 주소 찾기

§3 의 `TRACE_MATCH_ADDR0`, `TRACE_MATCH_ADDR1` 은 이 튜토리얼이 사용한 빌드 기준이라,
다음 경우에는 잘 맞지 않을 수 있습니다.

- 펌웨어를 직접 다시 빌드함
- 컴파일 옵션이 다른 펌웨어를 사용 중
- `times` 가 비어 있거나, 잡혔는데 위치가 이상함

이 셀은 `arm-none-eabi-objdump` 로 ELF 를 디스어셈블해서
**원하는 함수의 시작 주소**를 찾아 줍니다.

> 💡 `arm-none-eabi-objdump` 가 PATH 에 없으면 ChipWhisperer 빌드 환경(VM, Docker 등) 에서 실행하세요.


In [20]:
def find_function_addresses_from_elf(elf_path, function_names):
    """ELF 파일을 디스어셈블해 주어진 함수 이름들의 시작 주소를 찾는다."""
    result = subprocess.run(
        ["arm-none-eabi-objdump", "-d", elf_path],
        capture_output=True, text=True, check=True,
    )

    head_regex = re.compile(r"^([0-9a-fA-F]{8})\s<([^>]+)>:$")
    found = {}
    for line in result.stdout.splitlines():
        m = head_regex.match(line.strip())
        if m and m.group(2) in function_names:
            found[m.group(2)] = int(m.group(1), 16)
    return found


found = {}
import os.path

if not os.path.exists(FW_ELF_PATH):
    print(f"⚠️ ELF 경로를 찾을 수 없습니다: {FW_ELF_PATH}")
    print("   §6 의 빌드 셀을 먼저 실행했는지, FW_ELF_PATH 가 맞는지 확인하세요.")
else:
    try:
        found = find_function_addresses_from_elf(
            FW_ELF_PATH,
            ["SubBytes", "AddRoundKey", "ShiftRows", "MixColumns", "KeyExpansion"],
        )
        print("ELF 에서 찾은 함수 시작 주소:")
        for name, addr in found.items():
            print(f"  {name:<14s} = 0x{addr:08x}")
    except FileNotFoundError:
        # arm-none-eabi-objdump 자체가 PATH 에 없는 경우
        print("⚠️ arm-none-eabi-objdump 가 PATH 에 없습니다.")
        print("   ChipWhisperer 빌드 환경 (VM / Docker) 에서 다시 시도하세요.")
    except subprocess.CalledProcessError as e:
        print(f"⚠️ objdump 실행 실패: {e}")


ELF 에서 찾은 함수 시작 주소:
  KeyExpansion   = 0x080016f8
  AddRoundKey    = 0x080017a4
  SubBytes       = 0x080017d8
  ShiftRows      = 0x08001808


찾은 주소를 다시 PC 매칭에 반영하고 싶으면 아래 셀을 실행하세요.
이후 §12의 캡처 셀을 다시 실행하면 새 주소 기준으로 TraceWhisperer event가 잡힙니다.


In [21]:
if "found" in globals() and found:
    # 어떤 함수를 매칭할지 정한다 (필요 시 변경)
    target_funcs = ("SubBytes", "AddRoundKey")

    if all(name in found for name in target_funcs):
        new_addr0 = found[target_funcs[0]]
        new_addr1 = found[target_funcs[1]]
        scope.trace.set_isync_matches(addr0=new_addr0, addr1=new_addr1, match="both")

        # 전역 상수도 갱신해 두면 다음 셀 재실행 시 일관됨
        TRACE_MATCH_ADDR0 = new_addr0
        TRACE_MATCH_ADDR1 = new_addr1

        print("✅ ELF 기준 주소로 매칭 갱신 완료:")
        print(f"   {target_funcs[0]:<14s} = 0x{new_addr0:08x}")
        print(f"   {target_funcs[1]:<14s} = 0x{new_addr1:08x}")
    else:
        missing = [n for n in target_funcs if n not in found]
        print("필요한 함수 주소를 모두 찾지 못했습니다:", missing)
else:
    print("먼저 위 셀을 실행해 found 딕셔너리를 만드세요.")


✅ ELF 기준 주소로 매칭 갱신 완료:
   SubBytes       = 0x080017d8
   AddRoundKey    = 0x080017a4


---
## 15. 정리 — 연결 해제 / 빌드 산출물 청소

노트북을 마치면 **반드시 장비 연결을 해제**하세요.
그렇지 않으면 다음 세션에서 USB 연결 충돌이 발생할 수 있습니다.


In [22]:
try:
    scope.dis()
    target.dis()
    print("✅ 장비 연결 해제 완료")
    print("   다음 사용 시 §4 부터 다시 실행하세요.")
except Exception as e:
    print(f"⚠️ 연결 해제 중 오류: {e}")


✅ 장비 연결 해제 완료
   다음 사용 시 §4 부터 다시 실행하세요.


아래 셀은 **선택 사항**입니다.
직접 빌드한 산출물(`.txt` 디스어셈블 등)을 깨끗이 정리하고 싶을 때만 실행하세요.


In [23]:
isClean = True  # True 로 바꾸면 정리 수행

if isClean:
    !cd simpleserial-trace && make PLATFORM=$PLATFORM clean && rm -f simpleserial-trace-CW308_STM32F3.txt
    print("✅ 빌드 산출물 정리 완료")
else:
    print("isClean = False → 정리하지 않음")


No CRYPTO_TARGET passed - defaulting to TINYAES128C
Building for platform CW308_STM32F3 with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
rm -f -- simpleserial-trace-CW308_CC2538.hex simpleserial-trace-CW301_AVR.hex simpleserial-trace-CW303.hex simpleserial-trace-CW304.hex simpleserial-trace-CW308_MEGARF.hex simpleserial-trace-CW308_SAM4L.hex simpleserial-trace-CW308_STM32F0.hex simpleserial-trace-CW308_STM32F1.hex simpleserial-trace-CW308_STM32F2.hex simpleserial-trace-CW308_STM32F3.hex simpleserial-trace-CW308_STM32F4.hex simpleserial-trace-CW308_K24F.hex simpleserial-trace-CW308_NRF52.hex simpleserial-trace-CW308_AURIX.hex simpleserial-trace-CW308_SAML11.hex simpleserial-trace-CW308_EFM32TG11B.hex simpleserial-trace-CWLITEARM.hex simpleserial-trace-CWLITEXMEGA.hex simpleserial-trace-CWNANO.hex simpleserial-trace-CWHUSKY.hex simpleser

---
## 16. 자주 발생하는 문제와 해결 체크리스트

실습 중 막혔다면 가장 먼저 이 표를 확인하세요.

### A. 클럭 / 인터페이스 단계 오류

| 증상 | 원인 후보 | 해결 |
|------|----------|------|
| `fe_clock_alive == False` | 타깃 클럭이 Husky 에 도달하지 않음 | `scope.io.hs2 = "clkgen"` 확인, 20핀 커넥터 / 타깃 전원 확인 |
| `swo_clock_locked == False` | SWO 클럭 PLL 잠김 실패 | `SWO_TRIGGER_FREQ_MUL`, `SWO_ACPR` 를 기본값으로, 케이블 재확인, 타깃 리셋 |
| `AssertionError: SWO line not high` | TDO → D2 미연결 또는 SWD 전환 실패 | TDO→D2 점퍼 재연결, `reset_target()` 후 `jtag_to_swd()` 재실행 |
| `AssertionError: clock you chose doesn't seem to be active` | 타깃 클럭 미도달 | `scope.io.hs2 = "clkgen"` 확인, 20핀 커넥터 점검 |

### B. 캡처 / 매칭 단계 오류

| 증상 | 원인 후보 | 해결 |
|------|----------|------|
| `capture_trace()` 가 `None` 반환 | 타깃 응답 없음 | `reset_target(scope)` 후 재시도, 펌웨어가 `simpleserial-trace` 인지 확인 |
| `fifo_empty() == True` | trace event 자체가 발생 안 함 | `arm_trace()` 가 캡처 직전인지, PC 주소가 현재 ELF 와 일치하는지, trigger 구간 길이 확인 |
| 이벤트는 잡히는데 위치가 이상 | 함수 주소 불일치 | §14 ELF 절차로 주소 재확인, periodic PC sampling OFF 인지 확인 |
| 이벤트 수가 21이 아님 (예: 5, 18, 35) | SWO 대역폭 부족(드롭) 또는 다른 펌웨어 | `ACPR` 값을 늘려 SWO 속도 낮추기, 두 주소 중 하나만 모니터링, 펌웨어 재확인 |
| 첫 이벤트 timing σ 가 큼 | SWO 동기 불안정 | `swo_clock_locked` 재확인, 케이블 짧게/단단히 연결, 클럭 설정 유지 |

### C. 펌웨어 / 환경 오류

| 증상 | 원인 후보 | 해결 |
|------|----------|------|
| `EXPECTED_FW_HINT` assert 실패 | `simpleserial-trace` 가 아닌 펌웨어 | `DO_PROGRAM_TARGET = True`, `FW_HEX_PATH` 확인 후 §7 재실행 |
| `arm-none-eabi-objdump` not found | 호스트에 ARM 툴체인 없음 | ChipWhisperer 빌드 환경 (VM / Docker) 에서 실행 |
| 다음 실행 시 USB 연결 안 됨 | 직전 세션에서 `scope.dis()` 누락 | 보드 USB 재연결 후 §4 부터 다시 실행 |

### 일반 디버깅 팁
1. **무엇이든 막히면 일단 `reset_target(scope)`** 를 한 번 해 보세요. 70%는 이걸로 풀립니다.
2. SWD 모드 전환 직후에는 타깃이 잠깐 응답 불능일 수 있어, **리셋 후 buildtime 재확인**이 필수입니다.
3. 같은 펌웨어라도 다시 빌드하면 주소가 달라집니다. **항상 `*.elf` / `*.txt` 의 함수 주소를 의심**하세요.


---
## 17. 이 노트북의 학습 포인트 정리

### TraceWhisperer 설정 순서 (한눈에 보기)

```
 1. scope.trace.enabled = True                  # TraceWhisperer ON
 2. scope.trace.clock.fe_clock_src = ...        # front-end clock 선택
 3. scope.trace.trace_mode = "SWO"              # SWO 인터페이스 선택
 4. scope.trace.jtag_to_swd()                   # JTAG → SWD 전환
 5. SWO 클럭 파라미터 설정                       # ACPR / freq_mul / swo_div
 6. scope.trace.target_registers.DWT_CTRL = ... # sync 프레임 OFF
 7. scope.trace.capture.trigger_source = ...    # 트리거 소스 선택
 8. scope.trace.set_pattern_match(0, [3,8,32])  # 패턴 매칭 등록
    + capture.rules_enabled = [0]
 9. scope.trace.capture.mode = "while_trig"     # 캡처 지속 조건
10. scope.trace.set_isync_matches(addr0, addr1) # DWT.COMP 주소 등록
    + scope.trace.set_periodic_pc_sampling(0)   # 주기 샘플링 OFF
11. scope.trace.arm_trace()                     # arm
    + cw.capture_trace(...)                     # 동시 캡처
12. scope.trace.read_capture_data()             # raw 읽기
13. scope.trace.get_rule_match_times(...)       # timestamp 추출
```

### 꼭 가져갈 핵심 개념
1. **TraceWhisperer 는 전력 파형의 대체가 아니라 보조 도구**입니다. 시간 기준점을 줍니다.
2. **입문자는 raw trace 보다 pattern match + timestamp** 모드로 시작하는 편이 훨씬 쉽습니다.
3. 결국 **관심 함수의 정확한 주소를 매칭**하는 것이 trace 품질을 좌우합니다.
4. 펌웨어를 다시 빌드하면 **주소가 달라집니다.** ELF 에서 주소를 다시 찾는 습관이 중요합니다.
5. trace timestamp 와 ADC 샘플 단위가 다르므로, 시각화 시 **`× scope.clock.adc_mul`** 변환을 잊지 마세요.

---
실습이 잘 진행되었기를 바랍니다. 막히는 부분이 있으면 §16 의 체크리스트를 먼저 확인한 뒤,
ChipWhisperer 공식 문서 / 깃허브 이슈를 검색해 보세요.
